In [1]:
import loadFns as lf
import helperFns as hf

import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
from datetime import datetime 
from scipy import stats

import pickle

%load_ext autoreload
%autoreload 2

This code will walk you through creating a a pickle file that has all of the relevant information.

In [55]:
#load txt file -> read it line by line -> split each line by '_' -> first part is 'tag' and second part is 'sess_date' -> use this to loop till the all the lines in text
input_file_list = pd.DataFrame(columns=['tag', 'sess_date'])

count=0
with open('AS40_run_list_prac.txt', 'r') as file:
    for line in file:
        # Remove any leading/trailing whitespace including newline characters
        line = line.strip()
        #print (line)
        # Skip empty lines
        if not line:
            continue
        
        # Split the line by '_'
        parts = line.split('_')
        
        # Ensure the line splits into exactly two parts
        if len(parts) != 2:
            print(f"Skipping malformed line: {line}")
            continue
        
        tag_temp, sess_date_temp = parts

        date_obj = datetime.strptime(sess_date_temp, "%m%d%Y")
        converted_sess_date = date_obj.strftime("%y%m%d")
        #converted_sess_date = sess_date_temp ## for AS20 only

        input_file_list.loc[len(input_file_list)] = [tag_temp, converted_sess_date]

        count+=1


print(input_file_list[0:3])

sess_date_call = input_file_list['sess_date']

for i in range(len(sess_date_call)):  

        tag = input_file_list['tag'][i]
        sess_date = input_file_list['sess_date'][i]

        print (sess_date)

    tag sess_date
0  AS40    260122
260122


In [ ]:
def Interneuron_data():
    

In [72]:
def raw_data (tag, sess_date, neuronal_loc):
    '''
    Loads neuronal file, returns dataframe with trial-by-trial information

    Args:
        tag: often mouse ID + session type, string to look for in folder
        sess_id: date of session to analyze, in "YYYY-MM-DD_HR-MN-SC" format 
        neuronal_loc: relative path to folder that contains neuronal files

    Returns:
        cluster_group: 
        cluster_info:
        cluster_KSLabel:
        cluster_peak_channel:
        events:
        spike_clusters:
        spike_times_sec_adj:
        channel_map:

    '''

    #date_obj = datetime.strptime(sess_date, '%y%m%d')
    date_temp = date_obj.strftime('%m%d%Y')

    #date_temp = sess_date
    res = lf.find_folders(date_temp, neuronal_loc)

    res = [r for r in res if tag in r]
    res = [r for r in res if 'Single6Tone' in r] 

    res = sorted(res, key = len)
    temp = []
    for folder in res:
        if not any(os.path.commonpath([folder, top]) == top for top in temp):
            temp.append(folder)
    res = temp

    if not res:
        print(f"No matching KS folders found for mouse {tag} on {sess_date}, trying different date format:")
        date_temp_2 = date_obj.strftime('%m%d%y')
        res = lf.find_folders(date_temp_2, neuronal_loc)

        res = [r for r in res if tag in r]
        res = [r for r in res if 'Single6Tone' in r] 

        res = sorted(res, key = len)
        temp = []
        for folder in res:
            if not any(os.path.commonpath([folder, top]) == top for top in temp):
                temp.append(folder)
        res = temp

        if len(res) > 0:
            print("Folder(s) found with new date format.")
        
        if len(res) == 0:
            raise ValueError('No folders found for neuronal analysis.')

    if len(res) == 1:
        neuronal_loc = res[0]

    if len(res) > 1:
        print("Multiple folders found, please select one:")
        for idx, file in enumerate(res,1):
            print(f"{idx}. {os.path.basename(file)}")

        a = True
        while a == True:
            try:
                choice = int(input("Enter the number of the file you want to select: "))
                if 1 <= choice <= len(res):
                    neuronal_loc = res[choice - 1]
                    a = False
                else:
                    print("Invalid selection. Please enter a number from the list.")
            except ValueError:
                print("Invalid input. Please enter a number.")     

    spike_times = sorted(lf.find_files(".npy", "spike_times_sec_adj", neuronal_loc))
    neuronal_loc = os.path.abspath(os.path.join(os.path.dirname(spike_times[0]), '.'))
    spike_times = np.load(spike_times[0])

    cluster_group = sorted(lf.find_files(".tsv", "cluster_group", neuronal_loc))
    cluster_group = pd.read_csv(cluster_group[0], sep='\t')        

    cluster_info = sorted(lf.find_files(".tsv", "cluster_info", neuronal_loc))
    cluster_info = pd.read_csv(cluster_info[0], sep='\t')  

    cluster_KSLabel = sorted(lf.find_files(".tsv", "cluster_KSLabel", neuronal_loc))
    cluster_KSLabel = pd.read_csv(cluster_KSLabel[0], sep='\t')

    cluster_peak_channel = sorted(lf.find_files(".tsv", "cluster_peak_channel", neuronal_loc))
    cluster_peak_channel = pd.read_csv(cluster_peak_channel[0], sep='\t')

    event_times = sorted(lf.find_files(".csv", "events", neuronal_loc))
    event_times = pd.read_csv(event_times[0], delimiter=',', header=None)  

    print(event_times.shape)
    print(event_times.columns)

    spike_clusters = sorted(lf.find_files(".npy", "spike_clusters", neuronal_loc))
    spike_clusters = np.load(spike_clusters[0])      

    channel_map = sorted(lf.find_files(".npy", "channel_map", neuronal_loc))
    channel_map = np.load(channel_map[0])   

    print('raw_data_events:',event_times) 
    
    raw_data_df = {
        'cluster_group': cluster_group,
        'cluster_info': cluster_info,
        'cluster_KSLabel': cluster_KSLabel,
        'cluster_peak_channel': cluster_peak_channel,
        'event_times': event_times,
        'spike_clusters': spike_clusters,
        'spike_times': spike_times,
        'channel_map': channel_map
    }
    
    ##print (cluster_info['depth'])
    #raw_data_pd = pd.DataFrame(raw_data_df)

    #print(cluster_info.columns)
    #print(cluster_info.head())

    pkl_name = f'{tag}_{sess_date}_raw_data_imec1.pkl' 
    with open(pkl_name, 'wb') as f:
        pickle.dump(raw_data_df, f)
        

    return raw_data_df
    


In [ ]:
def make_pickle (trial_events, spikes_df, cluster_info, kept_clusters, raw_data_df, nb_times, tag, sess_date):

    uf = np.unique(trial_events['stim'])

    if len(uf)>8:
        session_type = 'testing_session'
    else:
        session_type = 'training_session'

    session_info = lf.get_row_df_from_public_sheet(tag, sess_date)

    all_clusters = np.unique(spikes_df['cluster'])

    edges_stim = np.arange(-1.0, 1.0, 0.005)
    tensor_stim = hf.gen_tensor(edges_stim, all_clusters, trial_events['stim_time'], spikes_df)

    edges_resp = np.arange(-1.0, 1.0, 0.005)
    tensor_resp = hf.gen_tensor(edges_resp, all_clusters, trial_events['resp_time'], spikes_df)

    event_times = trial_events['stim_time']

    stim_events_df = {
        'event' : 1 + 0*event_times, #just creating an array of 1 of the same length as event_times
        'time' : event_times
    }

    print ('event_times:', event_times)

    #all_beh_times_df = all_lick_per_side(tag, stim_events_df)
    channel_map = raw_data_df['channel_map']
    #print(raw_data_df['channel_map'])
    

    df_dict = {
        "all_session_info": session_info,
        "trial_events": trial_events,
        "spikes_df": spikes_df,
        "cluster_info": cluster_info, 
        "kept_clusters": kept_clusters, 
        "nb_times": nb_times,
        "tensor_stim": tensor_stim,
        "edges_stim": edges_stim,
        "tensor_resp": tensor_resp,
        "edges_resp": edges_resp, 
        "session_type":session_type,
        "uf": uf,
        #"right_lick_all": all_beh_times_df ['right_lick_time'],
        #"left_lick_all": all_beh_times_df ['left_lick_time'], 
        "channel_map": channel_map
        } 
    
    '''
        Change the pickle name based on imec1 vs imec0 ####
        '''   

    pkl_name = f'{tag}_{sess_date}_pickle_imec1.pkl' 
    with open(pkl_name, 'wb') as f:
        pickle.dump(df_dict, f)
        


In [ ]:
sess_date_call = input_file_list['sess_date']

bin_size = 0.005
steps = np.arange(-0.5, 1.0, bin_size)
xvals = (steps[0:-1] + steps[1:])/2

for i in range(len(sess_date_call)):  

        tag = input_file_list['tag'][i]
        sess_date = input_file_list['sess_date'][i]
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

        neuronal_location =  'H:/imec1_data/AS40_imec1'  # 'E:/Preprocessed_data/AS37' ##  # H:/imec1_data/AS38_imec1' # 'E:/Preprocessed_data/AS40' #'H:/imec1_data/AS38_imec1'' # 

        trial_events, spikes_df, cluster_info, kept_clusters, nb_times = lf.gen_dataframe_git(tag, sess_date, 
                                                                                interneuron_search = False, 
                                                                                neuronal_location = neuronal_location)
                
        raw_data_df = raw_data (tag, sess_date, neuronal_location)

        spikes_df_temp, stim_events_df, kept_clusters_temp, cluster_group_temp = lf.load_neuronal(tag, sess_date, 
                                                                                                  interneuron_search = False, 
                                                                                                  neuronal_loc = neuronal_location)
        
        date_n = datetime.strptime(sess_date, "%y%m%d").strftime("%m%d%y")
        all_beh_times_df = lf.all_lick_response (tag, sess_date, stim_events_df)
        #all_beh_times = pd.read_pickle('all_beh_times.pkl')

        make_pickle (trial_events, spikes_df, cluster_info, kept_clusters, raw_data_df, nb_times, tag, sess_date)

                                                                                                                

319 319
(1, 318)
Index([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,
       ...
       308, 309, 310, 311, 312, 313, 314, 315, 316, 317],
      dtype='int64', length=318)
raw_data_events:          0          1          2          3          4          5    \
0  19.040417  24.683413  33.185241  37.132137  41.047032  44.449896   

         6          7          8         9    ...          308          309  \
0  47.682086  50.988945  56.141244  59.89613  ...  1348.579064  1354.659422   

           310          311          312         313          314  \
0  1359.641046  1363.491938  1369.230909  1375.01258  1378.927477   

           315          316          317  
0  1384.869158  1391.930906  1397.819249  

[1 rows x 318 columns]
event_times: 0        19.040417
1        24.683413
2        33.185241
3        37.132137
4        41.047032
          ...     
313    1375.012580
314    1378.927477
315    1384.869158
316    1391.930906
317    1397.819249
Name: stim_time, Length: 318, dtyp

Summary session pickle

waveform session

In [76]:
with open("AS40_260122_pickle_imec1.pkl", "rb") as f:
    data = pickle.load(f)

# Print the contents
print(data)

print(type(data))

{'session_info':             06/25/2025                   ABR testing  NaN  NaN  NaN NaN  NaN  \
0  2026-01-22 00:00:00  Training_4tones + Recordings  NaN  NaN  241  25  90%   

         NaN  NaN  NaN  ...  NaN  NaN     NaN     NaN       NaN     NaN  \
0  55.0 uL/g  NaN  NaN  ...  87%  77%  150 ms  160 ms  160.0 uL  5.3 uL   

     NaN  NaN 27.6    2  
0  #REF!  NaN  NaN  NaN  

[1 rows x 47 columns], 'trial_events':        stim_time    resp_time       stim  cat  acc  dir  resp  corrT
0      19.040417          NaN  13.217465  1.0  2.0  0.0     0      0
1      24.683413          NaN  12.995225  1.0  2.0  0.0     0      0
2      33.185241    33.791292  12.550747  1.0  1.0  2.0     1      0
3      37.132137    37.310251  14.328661  3.0  1.0  1.0     1      0
4      41.047032    41.381577  12.550747  1.0  1.0  2.0     1      0
..           ...          ...        ...  ...  ...  ...   ...    ...
313  1375.012580  1375.996889  12.550747  1.0  1.0  2.0     1      0
314  1378.927477          N

In [77]:
# If it's a dictionary

#for key, value in AS40_260122_raw_data_imec1.items():
#    print(f"{key}: {value}")
#
## If a list
#
#for item in AS40_260122_raw_data_imec1:
#    print(item)

# If it's a pandas DataFrame: Some pickle files contain DataFrames:

df = pd.read_pickle("AS40_260122_raw_data_imec1.pkl")
print(df)
print(df.head())

{'cluster_group':      cluster_id  group
0             1   good
1             3   good
2             7   good
3             8   good
4            11  noise
..          ...    ...
125         303  noise
126         311  noise
127         315  noise
128         318  noise
129         333  noise

[130 rows x 2 columns], 'cluster_info':      cluster_id  Amplitude  ContamPct KSLabel  PT_ratio         amp  \
0             0    29669.6      100.0     mua  1.722051  536.207947   
1             1    38561.7      100.0    good  0.290100  728.015625   
2             2    19504.6      100.0     mua  1.611447  360.012604   
3             3    31019.8      100.0    good  0.269770  572.913757   
4             4    22509.1      100.0     mua  1.535975  423.645264   
..          ...        ...        ...     ...       ...         ...   
298         337    22412.6      100.0     mua  0.786099  352.305176   
299         338    33328.4      100.0     mua  0.652132  492.977356   
300         339    32281.1

AttributeError: 'dict' object has no attribute 'head'

In [50]:
# Pretty-print large objects

from pprint import pprint

with open("AS40_260122_raw_data_imec1.pkl", "rb") as f:
    AS40_260122_raw_data_imec1 = pickle.load(f)

pprint(AS40_260122_raw_data_imec1)

{'channel_map': array([[  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
         13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,
         26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,
         39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,
         52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,
         65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,
         78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,
         91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103,
        104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116,
        117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129,
        130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142,
        143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155,
        156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168,
        169, 170, 171, 172, 173, 17

In [ ]:
# Loading multiple objects from one pickle file

with open("AS40_260122_raw_data_imec1.pkl", "rb") as f:
    while True:
        try:
            obj = pickle.load(f)
            print(obj)
        except EOFError:
            break